# Plant Leaf Disease Multiclass Classification Using Deep Learning

This notebook trains a deep learning model to classify plant leaf diseases using the **PlantVillage dataset**.

The goal of this project is to automatically detect plant diseases from leaf images using **computer vision techniques**.

Model: ResNet18 (Transfer Learning)  
Framework: PyTorch  
Dataset: PlantVillage  

Classes used in this experiment:
- Potato Early Blight
- Potato Late Blight
- Potato Healthy
- Tomato Early Blight
- Tomato Late Blight
- Tomato Healthy

## 1. Google Drive Setup

Since this notebook runs on **Google Colab**, we mount Google Drive so the notebook can access the dataset and save trained models.

This allows us to store large datasets and model files outside the temporary Colab environment.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Import Required Libraries

In this section, we import all necessary libraries used in the project.

These include:
- **PyTorch** for building and training the deep learning model
- **Torchvision** for loading image datasets and image transformations
- **DataLoader utilities** for efficient batch training
- **Optimization tools** for updating model parameters during training

In [2]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

## 3. Define Dataset Path

Here we specify the location of the **PlantVillage dataset**.

The dataset is organized into folders where each folder represents a disease class.  
PyTorch’s `ImageFolder` automatically reads this structure and assigns labels to each class.

In [19]:
dataset_path = "/content/drive/MyDrive/888351/leaf_project/PlantVillage"

## 4. Image Preprocessing and Data Augmentation

Before feeding images into the neural network, they must be preprocessed.

For the training dataset, we apply **data augmentation techniques** to improve model generalization:
- Random horizontal flipping
- Random rotation
- Brightness and contrast adjustments

These transformations help the model learn robust features and reduce overfitting.

For validation and test datasets, only resizing and tensor conversion are applied.

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

## 5. Load Dataset

The PlantVillage dataset is loaded using PyTorch’s **ImageFolder** class.

ImageFolder automatically:
- scans the dataset directory
- assigns numerical labels to each class
- prepares images for model training.

We also display the number of images and the class names.

In [5]:
full_dataset = datasets.ImageFolder(root=dataset_path)
class_names = full_dataset.classes

print("Classes:", class_names)
print("Total images:", len(full_dataset))

Classes: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']
Total images: 6652


## 6. Split Dataset into Train, Validation, and Test Sets

To evaluate the model properly, the dataset is divided into three subsets:

- **Training Set (70%)** – used to train the model
- **Validation Set (15%)** – used to monitor model performance during training
- **Test Set (15%)** – used to evaluate the final model performance

This helps ensure the model generalizes well to unseen data.

In [6]:
train_dataset_full = datasets.ImageFolder(root=dataset_path, transform=train_transform)
eval_dataset_full = datasets.ImageFolder(root=dataset_path, transform=eval_transform)

In [7]:
dataset_size = len(full_dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

indices = torch.randperm(dataset_size).tolist()

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

In [20]:
from torch.utils.data import Subset

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(eval_dataset_full, val_indices)
test_dataset = Subset(eval_dataset_full, test_indices)

## 7. Create DataLoaders

DataLoaders are used to load data in **mini-batches** during training.

Batch training improves:
- computational efficiency
- GPU utilization
- memory management

Shuffling is applied to the training set to prevent the model from learning the order of the data.

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 8. Model Architecture

In this project, we use **ResNet18**, a convolutional neural network pretrained on the ImageNet dataset.

Transfer learning is applied by:
- freezing the pretrained feature extraction layers
- replacing the final fully connected layer to match the number of plant disease classes

This approach allows the model to leverage previously learned visual features.

In [10]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(class_names))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


## 9. Device Configuration

Deep learning models can run on either CPU or GPU.

This section checks whether a **GPU is available**.  
If a GPU is available, the model will use CUDA for faster training; otherwise, it will run on CPU.

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Using device:", device)

Using device: cuda


## 10. Define Loss Function and Optimizer

To train the model, we define:

**Loss Function**
- CrossEntropyLoss is used for multiclass classification problems.

**Optimizer**
- Adam optimizer is used to update model parameters based on the computed gradients.

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

## 11. Validation Evaluation

After each training epoch, the model is evaluated on the validation dataset.

Validation performance helps detect overfitting and ensures the model generalizes well to unseen images.

The model with the **best validation accuracy** is saved for final testing.

In [13]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

## 12. Model Training

The model is trained for several epochs.

During each epoch:
1. Images are passed through the network (forward pass)
2. The prediction error is computed using the loss function
3. Gradients are calculated using backpropagation
4. Model parameters are updated by the optimizer

Training accuracy and loss are tracked to monitor model performance.

In [14]:
num_epochs = 5
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
    print("-" * 50)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

Epoch [1/5]
Train Loss: 0.7811 | Train Acc: 76.35%
Val   Loss: 0.3848 | Val   Acc: 90.97%
--------------------------------------------------
Epoch [2/5]
Train Loss: 0.3813 | Train Acc: 89.20%
Val   Loss: 0.2680 | Val   Acc: 92.18%
--------------------------------------------------
Epoch [3/5]
Train Loss: 0.3047 | Train Acc: 91.73%
Val   Loss: 0.2228 | Val   Acc: 93.58%
--------------------------------------------------
Epoch [4/5]
Train Loss: 0.2634 | Train Acc: 92.29%
Val   Loss: 0.2087 | Val   Acc: 93.48%
--------------------------------------------------
Epoch [5/5]
Train Loss: 0.2334 | Train Acc: 93.00%
Val   Loss: 0.1831 | Val   Acc: 94.18%
--------------------------------------------------


## 13. Final Model Evaluation

After training is complete, the best model is evaluated on the **test dataset**.

This provides an unbiased estimate of how well the model performs on new, unseen images.

In [15]:
model.load_state_dict(best_model_wts)

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")

Best Validation Accuracy: 94.18%
Test Accuracy: 93.69%


## 14. Save the Trained Model

The trained model is saved as a `.pth` file.

This file contains the learned model parameters and can later be loaded for inference in the **Streamlit web application**.

In [18]:
save_path = "/content/drive/MyDrive/888351/leaf_project/leaf_model_v2.pth"
torch.save(model.state_dict(), save_path)
print("Saved improved model to:", save_path)

Saved improved model to: /content/drive/MyDrive/888351/leaf_project/leaf_model_v2.pth
